# Karachi WaterWise - Member 3
**Random Forest, K-Means clustering and feature importance**

How to run: in Google Colab choose *Runtime > Run all*. When the upload window appears, select **both** files from Member 1:
`cleaned_full_dataset.csv` and `member1_coping_segmentation_dataset.csv`.
At the end, `Member3_outputs.zip` is created and downloaded (model, results, graphs).

Final classifier features (7): Household_Members, House_Type, Income_Level_After_Pandemic, Piped_Supply_Days_Per_Month, Fixed_Supply_Timing, Has_Underground_Tank, Has_Overhead_Tank.
`Monthly_Tanker_Expense` is NOT used for classification because it is a consequence of water scarcity (target leakage).

In [ ]:
# ===== Member 3 | Setup: libraries, files and output folder =====
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = "Member3_outputs"      # our own output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Are we running inside Google Colab?
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# In Colab: upload BOTH of Member 1's files in one go (select both in the upload window):
#   1) cleaned_full_dataset.csv                   (18 columns)
#   2) member1_coping_segmentation_dataset.csv    (11 columns)
if IN_COLAB:
    files.upload()

def find_newest_file(keyword):
    """Return the most recently added CSV in this folder whose name contains the keyword."""
    matches = [name for name in os.listdir(".") if keyword in name and name.endswith(".csv")]
    if not matches:
        raise FileNotFoundError("No CSV with '" + keyword + "' in its name was found. Please upload it.")
    return max(matches, key=os.path.getmtime)

DATA_FILE = find_newest_file("cleaned_full_dataset")        # used for Random Forest
DATA_FILE_SEG = find_newest_file("coping_segmentation")     # used for K-Means
print("Classification data file:", DATA_FILE)
print("Segmentation data file:  ", DATA_FILE_SEG)

# Safety check: make sure these are Member 1's cleaned files (not the raw survey file)
check_clean = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
check_seg = pd.read_csv(DATA_FILE_SEG, encoding="utf-8-sig")
assert "Is_Renter" in check_clean.columns and "Water_Scarcity_Target" in check_clean.columns, \
    "Wrong file: this is not Member 1's cleaned_full_dataset.csv"
assert "Uses_Tanker" in check_seg.columns and "Water_Scarcity_Target" in check_seg.columns, \
    "Wrong file: this is not Member 1's member1_coping_segmentation_dataset.csv"
assert len(check_clean) == len(check_seg), "The two files have a different number of rows"
print("Both files look correct. Rows:", len(check_clean))

## Part 1 - Random Forest
### Step 4a: Load data, select features, split

In [ ]:
# ===== Member 3 | Step 4a: Load data, select features, train/test split =====
import pandas as pd
from sklearn.model_selection import train_test_split

# The team's agreed 7 features (tanker expense is excluded because it causes target leakage)
FEATURES = [
    "Household_Members",
    "House_Type",
    "Income_Level_After_Pandemic",
    "Piped_Supply_Days_Per_Month",
    "Fixed_Supply_Timing",
    "Has_Underground_Tank",
    "Has_Overhead_Tank",
]
TARGET = "Water_Scarcity_Target"   # 1 = scarcity reported, 0 = no scarcity

# 1) Load the data
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")
print("Dataset shape:", df.shape)

# 2) Check that all required columns exist in the file
missing_columns = [col for col in FEATURES + [TARGET] if col not in df.columns]
print("Missing columns:", missing_columns)

# 3) Separate the inputs (X) from the answer we want to predict (y)
X = df[FEATURES].copy()
y = df[TARGET].copy()

# 4) Check missing values and class counts
print("\nMissing values in each feature:")
print(X.isnull().sum())
print("\nTarget counts:")
print(y.value_counts())

# 5) 80% training / 20% testing (same settings as Member 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,   # same random split every time we run the code
    stratify=y,        # keep the 0/1 ratio the same in train and test
)

print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))
print("\nTrain class share:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class share:")
print(y_test.value_counts(normalize=True).round(3))


### Step 4b: Preprocessing pipeline and first Random Forest

In [ ]:
# ===== Member 3 | Step 4b: Preprocessing pipeline + first Random Forest =====
# (Run this after Step 4a: X_train, X_test, y_train, y_test must already exist)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

# 1) Split the columns into two groups
categorical_features = ["House_Type", "Fixed_Supply_Timing"]       # text columns
numeric_features = [col for col in FEATURES if col not in categorical_features]
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# 2) Numeric columns: fill empty cells with the median, then scale
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 3) Text columns: fill empty cells with the most common value, then convert to 0/1 columns
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# 4) Combine both into one preprocessor
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

# 5) Preprocessor + Random Forest in one pipeline (first model, default settings)
rf_model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,          # 100 trees
        class_weight="balanced",   # give the smaller class (No scarcity) equal importance
        random_state=42,
    )),
])

# 6) Learn from the training data only
rf_model.fit(X_train, y_train)
print("\nRandom Forest trained!")

forest = rf_model.named_steps["classifier"]
print("Number of trees:", len(forest.estimators_))
print("Input columns after encoding:", len(rf_model.named_steps["preprocessing"].get_feature_names_out()))

# 7) See how the trees vote: take the first household in the training data
one_household = X_train.iloc[[0]]
one_processed = rf_model.named_steps["preprocessing"].transform(one_household)
tree_votes = [int(tree.predict(one_processed)[0]) for tree in forest.estimators_]

print("\nHousehold:")
print(one_household.to_string(index=False))
print("Trees voting 'Scarcity' (1):   ", sum(tree_votes))
print("Trees voting 'No scarcity' (0):", len(tree_votes) - sum(tree_votes))
print("Forest probability of scarcity:", round(rf_model.predict_proba(one_household)[0, 1], 3))
print("Forest final prediction:", rf_model.predict(one_household)[0])


### Step 4c: Stratified 5-fold cross-validation

In [ ]:
# ===== Member 3 | Step 4c: Stratified 5-fold cross-validation for Random Forest =====
# (Run after Step 4b: rf_model, X_train and y_train must already exist)
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

# 1) Same CV settings as Member 2, so the comparison is fair
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2) Metrics we want to measure in every fold
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

# 3) Run CV on the TRAINING data only (the test set stays untouched).
#    The pipeline is re-fitted inside every fold, so preprocessing never sees the validation fold.
cv_results = cross_validate(
    rf_model, X_train, y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,   # also measure the score on the training part of each fold
)

# 4) F1 score of each fold
print("F1 per fold (validation):", cv_results["test_f1"].round(3))

# 5) Summary table: average and standard deviation over the 5 folds
summary = pd.DataFrame({
    "Mean": [cv_results["test_" + name].mean() for name in scoring],
    "Std":  [cv_results["test_" + name].std() for name in scoring],
}, index=list(scoring.keys())).round(3)
print("\nRandom Forest 5-fold CV (validation folds):")
print(summary)

# 6) Overfitting check: compare training F1 with validation F1
print("\nAverage F1 on training part of folds:  ", round(cv_results["train_f1"].mean(), 3))
print("Average F1 on validation part of folds:", round(cv_results["test_f1"].mean(), 3))


### Step 6: Hyperparameter tuning

In [ ]:
# ===== Member 3 | Step 6: Hyperparameter tuning for Random Forest =====
# (Run after Step 4c: rf_model, cv, X_train and y_train must already exist)
import pandas as pd
from sklearn.model_selection import GridSearchCV

# 1) The settings we want to try ("classifier__" means: a setting of the Random Forest step)
param_grid = {
    "classifier__n_estimators": [100, 200, 300],      # number of trees
    "classifier__max_depth": [3, 5, 7, None],         # how deep each tree can grow (None = no limit)
    "classifier__min_samples_leaf": [1, 2, 5],        # minimum households in a leaf
}

# 2) Try every combination with 5-fold CV on the TRAINING data (test set stays untouched)
rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=cv,                     # same StratifiedKFold as before
    scoring="f1",
    return_train_score=True,   # lets us check overfitting for each combination
    n_jobs=-1,                 # use all CPU cores
)
rf_grid.fit(X_train, y_train)

# 3) Best settings found
print("Best parameters:", rf_grid.best_params_)
print("Best CV F1:", round(rf_grid.best_score_, 3))

# 4) Top 5 combinations
results = pd.DataFrame(rf_grid.cv_results_)
top5 = results.sort_values("rank_test_score").head(5)[
    ["param_classifier__n_estimators", "param_classifier__max_depth",
     "param_classifier__min_samples_leaf",
     "mean_train_score", "mean_test_score", "std_test_score"]
].round(3)
print("\nTop 5 combinations:")
print(top5.to_string(index=False))

# 5) The best model (already re-trained on the full training data)
best_rf = rf_grid.best_estimator_


### Step 7: Final evaluation on the test set (used only once)

In [ ]:
# ===== Member 3 | Step 7: Final evaluation of the tuned Random Forest on the test set =====
# (Run after Step 6: best_rf, X_test and y_test must already exist.
#  The test set is used ONLY ONCE here. Do not change any settings after looking at these results.)
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
)

# 1) Predict the 188 test households the model has never seen
y_pred = best_rf.predict(X_test)                    # predicted class: 0 or 1
y_prob = best_rf.predict_proba(X_test)[:, 1]        # predicted probability of scarcity

# 2) Calculate the metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Random Forest": [accuracy, precision, recall, f1, roc_auc],
}).round(3)
print("RANDOM FOREST - TEST SET RESULTS")
print(results.to_string(index=False))

# 3) Detailed report for both classes
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No scarcity", "Scarcity"]))

# 4) Confusion matrix (rows = actual, columns = predicted)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No scarcity", "Scarcity"],
            yticklabels=["No scarcity", "Scarcity"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix (test set)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "member3_rf_confusion_matrix.png"), dpi=200)
plt.show()

# 5) Save the results table (same layout as Member 2's comparison table)
results.to_csv(os.path.join(OUTPUT_DIR, "member3_rf_results.csv"), index=False)
print("\nSaved files:", sorted(os.listdir(OUTPUT_DIR)))


### Feature importance

In [ ]:
# ===== Member 3 | Feature importance of the tuned Random Forest =====
# (Run after Step 7: best_rf, X_test, y_test, FEATURES and OUTPUT_DIR must already exist.
#  We only EXPLAIN the model here. We do not change it.)
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# 1) Built-in importance (impurity based): how much each column helped the trees to split.
#    It is calculated for the encoded columns, so we add the pieces of one feature back together.
preprocessing = best_rf.named_steps["preprocessing"]
forest = best_rf.named_steps["classifier"]
encoded_names = preprocessing.get_feature_names_out()
encoded_importance = pd.Series(forest.feature_importances_, index=encoded_names)

builtin_importance = {}
for feature in FEATURES:
    # e.g. "categorical__House_Type_Apartment" belongs to the feature "House_Type"
    pieces = [value for name, value in encoded_importance.items()
              if name.split("__", 1)[1].startswith(feature)]
    builtin_importance[feature] = sum(pieces)

# 2) Permutation importance: shuffle ONE column at a time and see how much the F1 score drops.
#    A big drop means the model depends on that column. We use the test set (188 households).
perm = permutation_importance(
    best_rf, X_test, y_test,
    scoring="f1",
    n_repeats=30,          # repeat the shuffling 30 times to get a stable average
    random_state=42,
)

importance_table = pd.DataFrame({
    "Feature": FEATURES,
    "Builtin_Importance": [builtin_importance[f] for f in FEATURES],
    "Permutation_F1_Drop": perm.importances_mean,
    "Permutation_Std": perm.importances_std,
}).sort_values("Permutation_F1_Drop", ascending=False).round(3)
print(importance_table.to_string(index=False))

# 3) Bar chart of the permutation importance
plt.figure(figsize=(8, 4.5))
plt.barh(importance_table["Feature"], importance_table["Permutation_F1_Drop"],
         xerr=importance_table["Permutation_Std"], color="steelblue")
plt.gca().invert_yaxis()                      # most important feature on top
plt.xlabel("Drop in F1 score when the feature is shuffled")
plt.title("Random Forest - Permutation Feature Importance (test set)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "member3_feature_importance.png"), dpi=200)
plt.show()

# 4) Save the table
importance_table.to_csv(os.path.join(OUTPUT_DIR, "member3_feature_importance.csv"), index=False)
print("\nSaved:", sorted(f for f in os.listdir(OUTPUT_DIR) if "importance" in f))


## Part 2 - K-Means clustering (separate unsupervised analysis)
### Step 8: Prepare the data

In [ ]:
# ===== Member 3 | Step 8: Prepare the data for K-Means clustering =====
# (DATA_FILE_SEG was set in the setup cell at the top of this notebook.)
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1) Load Member 1's segmentation file (we only read it, we never modify it)
df_seg = pd.read_csv(DATA_FILE_SEG, encoding="utf-8-sig")
print("Segmentation data shape:", df_seg.shape)

# 2) Features used for clustering (K-Means never sees the target column)
cluster_features = [
    "Monthly_Tanker_Expense",
    "Has_Piped_Connection",
    "Piped_Supply_Days_Per_Month",
    "Has_Underground_Tank",
    "Has_Overhead_Tank",
    "Household_Members",
    "Income_Level_After_Pandemic",
]
# Kept ONLY for describing the clusters after they are built (not used to build them)
profile_target = df_seg["Water_Scarcity_Target"].copy()

X_cluster = df_seg[cluster_features].copy()

# 3) Tanker expense is very skewed (many 0s, a few very large values),
#    so we use log(1 + expense) to stop a few big values from dominating the distances
X_cluster["Monthly_Tanker_Expense"] = np.log1p(X_cluster["Monthly_Tanker_Expense"])

# 4) Check for missing values (K-Means cannot work with empty cells)
print("\nMissing values per feature:")
print(X_cluster.isnull().sum())

# 5) Scale every feature to mean 0 and standard deviation 1,
#    because K-Means uses distances and big-number features would otherwise dominate
cluster_scaler = StandardScaler()
X_scaled = cluster_scaler.fit_transform(X_cluster)

print("\nScaled data shape:", X_scaled.shape)
print("Mean of each scaled feature (should be about 0):", X_scaled.mean(axis=0).round(2))
print("Std of each scaled feature (should be about 1): ", X_scaled.std(axis=0).round(2))


### Step 9: Choose K (Elbow Method and Silhouette Score)

In [ ]:
# ===== Member 3 | Step 9: Choose K with the Elbow Method and Silhouette Score =====
# (Run after Step 8: X_scaled must already exist)
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

OUTPUT_DIR = "Member3_outputs"            # our own output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) Try K = 2 to 10 and record inertia and silhouette score for each K
k_values = list(range(2, 11))
inertias = []
silhouettes = []

for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)                       # distance of points to their cluster centres
    silhouettes.append(silhouette_score(X_scaled, labels)) # how well-separated the clusters are

k_table = pd.DataFrame({
    "K": k_values,
    "Inertia": inertias,
    "Silhouette": silhouettes,
}).round(3)
print(k_table.to_string(index=False))

# 2) Plot both curves side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(k_values, inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("Inertia (lower = tighter clusters)")

axes[1].plot(k_values, silhouettes, marker="o", color="darkorange")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette score (higher = better separated)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "member3_kmeans_elbow_silhouette.png"), dpi=200)
plt.show()

# 3) K with the highest silhouette score (a guide, not the final decision)
best_k_by_silhouette = k_table.loc[k_table["Silhouette"].idxmax(), "K"]
print("K with the highest silhouette score:", int(best_k_by_silhouette))


### Step 10: Final K-Means (K=5), visualisation and cluster profiles

In [ ]:
# ===== Member 3 | Step 10: Final K-Means (K=5), visualisation and cluster profiles =====
# (Run after Step 9: X_scaled, df_seg, profile_target and OUTPUT_DIR must already exist)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

K = 5   # chosen in Step 9 (elbow, silhouette and stability check)

# 1) Final K-Means model
kmeans_final = KMeans(n_clusters=K, n_init=10, random_state=42)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# 2) Profile table: describe every cluster using the ORIGINAL (unscaled) values.
#    The scarcity target was NOT used to build the clusters, we only look at it here.
profile_data = df_seg.copy()
profile_data["Cluster"] = cluster_labels

cluster_profile = profile_data.groupby("Cluster").agg(
    Households=("Cluster", "size"),
    Scarcity_Percent=("Water_Scarcity_Target", lambda s: s.mean() * 100),
    Tanker_Users_Percent=("Uses_Tanker", lambda s: s.mean() * 100),
    Median_Tanker_Expense_PKR=("Monthly_Tanker_Expense", "median"),
    Piped_Connection_Percent=("Has_Piped_Connection", lambda s: s.mean() * 100),
    Avg_Piped_Days_Per_Month=("Piped_Supply_Days_Per_Month", "mean"),
    Underground_Tank_Percent=("Has_Underground_Tank", lambda s: s.mean() * 100),
    Overhead_Tank_Percent=("Has_Overhead_Tank", lambda s: s.mean() * 100),
    Avg_Household_Members=("Household_Members", "mean"),
    Avg_Income_Level=("Income_Level_After_Pandemic", "mean"),
).round(1)
print(cluster_profile.T.to_string())

# 3) Draw the clusters in 2D (PCA only squeezes 7 features into 2 axes for plotting)
pca = PCA(n_components=2, random_state=42)
points_2d = pca.fit_transform(X_scaled)
print("\nVariance explained by the 2 PCA axes:", (pca.explained_variance_ratio_.sum() * 100).round(1), "%")

plt.figure(figsize=(8, 6))
sns.scatterplot(x=points_2d[:, 0], y=points_2d[:, 1], hue=cluster_labels,
                palette="tab10", s=40, alpha=0.8)
plt.title("Household clusters (K-Means, K=5) shown with PCA")
plt.xlabel("PCA axis 1")
plt.ylabel("PCA axis 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "member3_kmeans_clusters_pca.png"), dpi=200)
plt.show()

# 4) Save the results. row_id = row number in Member 1's file (same order as cleaned_full_dataset.csv)
assignments = pd.DataFrame({"row_id": range(len(cluster_labels)), "cluster": cluster_labels})
assignments.to_csv(os.path.join(OUTPUT_DIR, "member3_cluster_assignments.csv"), index=False)
cluster_profile.to_csv(os.path.join(OUTPUT_DIR, "member3_cluster_profiles.csv"))
print("\nSaved files:", sorted(os.listdir(OUTPUT_DIR)))


## Part 3 - Final outputs for the team
Saves the model, checks that it loads and predicts, and creates the zip file.

In [ ]:
# ===== Member 3 | Final outputs: save the model and check it works like Member 4 will use it =====
# (Run after Step 7: best_rf, rf_grid, results, X, X_test, FEATURES, TARGET and OUTPUT_DIR must exist)
import os
import json
import joblib
import shutil
import sklearn
import pandas as pd

# 1) Save the complete pipeline (preprocessing + Random Forest) in one file
model_path = os.path.join(OUTPUT_DIR, "member3_random_forest.joblib")
joblib.dump(best_rf, model_path)

# 2) Save a small information file for the team (Member 4 needs the library version and the input format)
model_info = {
    "model": "RandomForestClassifier inside a scikit-learn Pipeline (preprocessing + classifier)",
    "scikit_learn_version": sklearn.__version__,
    "target": TARGET,
    "target_meaning": "1 = household reported water scarcity, 0 = no scarcity",
    "input_features": FEATURES,
    "allowed_values_House_Type": sorted(X["House_Type"].unique().tolist()),
    "allowed_values_Fixed_Supply_Timing": sorted(X["Fixed_Supply_Timing"].unique().tolist()),
    "best_parameters": rf_grid.best_params_,
    "cv_f1_score": round(float(rf_grid.best_score_), 3),
    "test_set_results": {k: float(v) for k, v in results.set_index("Metric")["Random Forest"].items()},
}
with open(os.path.join(OUTPUT_DIR, "member3_model_info.json"), "w") as file:
    json.dump(model_info, file, indent=2)

# 3) Check: load the saved file again and compare with the original model
loaded_model = joblib.load(model_path)
same_predictions = (loaded_model.predict(X_test) == best_rf.predict(X_test)).all()
print("Loaded model gives the same predictions as the original:", same_predictions)

# 4) Check: predict ONE new household from raw values (this is what the app will do)
new_household = pd.DataFrame([{
    "Household_Members": 7,
    "House_Type": model_info["allowed_values_House_Type"][0],
    "Income_Level_After_Pandemic": 4,
    "Piped_Supply_Days_Per_Month": 2,
    "Fixed_Supply_Timing": "No",
    "Has_Underground_Tank": 1,
    "Has_Overhead_Tank": 1,
}])
print("\nExample household:")
print(new_household.to_string(index=False))
print("Predicted class:", loaded_model.predict(new_household)[0])
print("Probability of scarcity:", round(loaded_model.predict_proba(new_household)[0, 1], 3))

# 5) List everything we created
print("\nFiles in", OUTPUT_DIR, ":")
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", name)

# 6) Put everything into one zip file to download
shutil.make_archive("Member3_outputs", "zip", OUTPUT_DIR)
print("\nCreated Member3_outputs.zip")
if IN_COLAB:
    files.download("Member3_outputs.zip")     # downloads the zip to your computer


## Optional - leakage demonstration

In [ ]:
# ===== OPTIONAL | Leakage demonstration (side experiment, NOT the final model) =====
# Shows how much the scores change if the tanker-expense column is added back.
# Do not change the final model or its settings based on this result.
from sklearn.base import clone
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

FEATURES_LEAK = FEATURES + ["Monthly_Tanker_Expense"]    # the 7 features + tanker expense
X_leak = df[FEATURES_LEAK]
Xl_train, Xl_test = X_leak.loc[X_train.index], X_leak.loc[X_test.index]   # same rows as before

# Same preprocessing as before, but with the extra numeric column
num_l = [c for c in FEATURES_LEAK if c not in categorical_features]
pre_l = ColumnTransformer([("numeric", numeric_pipeline, num_l),
                           ("categorical", categorical_pipeline, categorical_features)])
rf_leak = Pipeline([("preprocessing", pre_l),
                    ("classifier", clone(best_rf.named_steps["classifier"]))])   # same tuned settings

for name, model, Xtr, Xte in [("Without tanker expense (final)", best_rf, X_train, X_test),
                              ("With tanker expense (leakage)", rf_leak, Xl_train, Xl_test)]:
    cv_f1 = cross_val_score(clone(model), Xtr, y_train, cv=cv, scoring="f1").mean()
    pred = clone(model).fit(Xtr, y_train).predict(Xte)
    print(f"{name:34s} CV F1 {cv_f1:.3f} | test F1 {f1_score(y_test, pred):.3f} | test accuracy {accuracy_score(y_test, pred):.3f}")